# Diabetes Risk Prediction
## Model Training: Random Forest and XGBoost
This notebook trains Random Forest and XGBoost on the preprocessed data from the EDA phase. We start with the raw unbalanced data as our baseline.

### Load the Unscaled Training Data

In [ ]:
# Load preprocessed data
import numpy as np
import pandas as pd

X_train = np.load('data/X_train.npy')
X_test = np.load('data/X_test.npy')
y_train = np.load('data/y_train.npy')
y_test = np.load('data/y_test.npy')
feature_names = pd.read_csv('data/feature_names.csv').values.flatten()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Feature names:", feature_names)

X_train: (202944, 21)
y_train: (202944,)
Feature names: ['HighBP' 'HighChol' 'CholCheck' 'BMI' 'Smoker' 'Stroke'
 'HeartDiseaseorAttack' 'PhysActivity' 'Fruits' 'Veggies'
 'HvyAlcoholConsump' 'AnyHealthcare' 'NoDocbcCost' 'GenHlth' 'MentHlth'
 'PhysHlth' 'DiffWalk' 'Sex' 'Age' 'Education' 'Income']


### Random Forest on Raw Training Data

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)

# Results
print("Random Forest - Raw Data Results:\n")
print(classification_report(y_test, rf_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]), 4))

Random Forest - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.96      0.91     42741
    Diabetes       0.52      0.21      0.30      7995

    accuracy                           0.84     50736
   macro avg       0.69      0.59      0.61     50736
weighted avg       0.81      0.84      0.82     50736

AUC-ROC: 0.7921


#### Random Forest - Raw Data Interpretation

The results confirm the class imbalance problem we identified during EDA. The model achieves 84% accuracy, but this is misleading because it's mostly just correctly predicting healthy patients (96% recall) while only catching 21% of actual diabetic patients.

Key takeaways:
- The model misses roughly 4 out of every 5 diabetic patients (recall = 0.21)
- When it does flag someone as diabetic, it's only right about half the time (precision = 0.52)
- The F1-score for diabetes is 0.30, which is poor
- AUC-ROC of 0.79 shows the model has some ability to distinguish between classes but struggles significantly with the minority class

This baseline result demonstrates exactly why balancing techniques like SMOTE and SMOTE-ENN are needed. Without addressing the imbalance, the model effectively learns to default to predicting "no diabetes" because that's the safe bet given the 84/16 class split.

### XGBoost on Raw Training Data
Training XGBoost with default hyperparameters on the raw unbalanced dataset.
XGBoost builds trees sequentially, where each new tree focuses on correcting the mistakes of the previous ones.

In [7]:
from xgboost import XGBClassifier

# Train XGBoost
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_predictions = xgb_model.predict(X_test)

# Results
print("XGBoost - Raw Data Results:\n")
print(classification_report(y_test, xgb_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]), 4))

XGBoost - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.97      0.92     42741
    Diabetes       0.57      0.21      0.30      7995

    accuracy                           0.85     50736
   macro avg       0.72      0.59      0.61     50736
weighted avg       0.82      0.85      0.82     50736

AUC-ROC: 0.8194


#### XGBoost - Raw Data Interpretation

XGBoost performs very similarly to Random Forest on the raw data. Diabetes recall is identical at 0.21 and F1-score is the same at 0.30. XGBoost shows slight improvement in precision (0.57 vs 0.52) and AUC-ROC (0.82 vs 0.79), suggesting it ranks patients slightly better overall.

The key finding here is that switching to a more powerful model alone does not solve the class imbalance problem. Both models default to predicting "no diabetes" for the vast majority of cases. This confirms that the issue lies in the data distribution, not the model architecture, and motivates the need for resampling techniques like SMOTE and SMOTE-ENN.